# 01. 인벤토리 + 스키마 발견

**목표**: 전체 [DONE] 데이터 현황 파악 + 각 데이터 타입의 필드 구조 발견

**산출물**: `eda_output/phase1_inventory.json`, `eda_output/phase2_schema.json`

In [ ]:
# ── 환경 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

# backend/ 루트를 path에 추가
BACKEND_DIR = Path.cwd().parent.parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import plotly.express as px
import plotly.graph_objects as go
from tqdm.auto import tqdm

from scripts.eda.common import (
    DATA_DIR,
    count_records,
    count_records_fast,
    detect_root_type,
    discover_done_files,
    get_file_size_mb,
    get_sample,
    infer_field_types,
    save_result,
)
from scripts.eda.data_registry import (
    CATEGORIES,
    get_agency_name,
    get_all_files,
)

# plotly 기본 테마
import plotly.io as pio
pio.templates.default = "plotly_white"

print(f"DATA_DIR: {DATA_DIR}")
print(f"카테고리 수: {len(CATEGORIES)}")

## 1. 파일 인벤토리

47~48개 `[DONE]` JSON 파일을 스캔하여 크기, 레코드 수, 루트 타입을 파악합니다.

In [ ]:
# ── 파일 인벤토리 수집 ─────────────────────────────────────
done_files = discover_done_files()
print(f"발견된 [DONE] 파일 수: {len(done_files)}")

# 카테고리 매핑 생성
all_file_map = {item["file"]: item["category"] for item in get_all_files()}

inventory = []
for finfo in tqdm(done_files, desc="파일 스캔"):
    filepath = Path(finfo["path"])
    root_type = detect_root_type(filepath)
    category = all_file_map.get(finfo["name"], "unknown")
    cat_label = CATEGORIES.get(category, {}).get("label", "미분류")
    agency = get_agency_name(finfo["name"])

    inventory.append({
        "filename": finfo["name"],
        "size_mb": finfo["size_mb"],
        "root_type": root_type,
        "category": category,
        "category_label": cat_label,
        "agency": agency,
    })

print(f"\n총 파일: {len(inventory)}개")
print(f"총 크기: {sum(f['size_mb'] for f in inventory):.1f} MB")

In [ ]:
# ── 레코드 수 카운트 (시간 소요) ──────────────────────────
# 대용량 파일은 스트리밍 카운트, 소용량은 json.load
for item in tqdm(inventory, desc="레코드 카운트"):
    filepath = DATA_DIR / item["filename"]
    item["record_count"] = count_records_fast(filepath)

total_records = sum(f["record_count"] for f in inventory)
print(f"\n총 레코드: {total_records:,}건")

## 2. 시각화: 카테고리별 파일 크기

In [ ]:
# ── 카테고리별 파일 크기 수평 바 차트 ─────────────────────
import pandas as pd

df_inv = pd.DataFrame(inventory)
df_cat_size = (
    df_inv.groupby(["category", "category_label"])
    .agg(total_size_mb=("size_mb", "sum"), file_count=("filename", "count"))
    .reset_index()
    .sort_values("total_size_mb", ascending=True)
)

fig = px.bar(
    df_cat_size,
    x="total_size_mb",
    y="category_label",
    orientation="h",
    title="카테고리별 총 파일 크기 (MB)",
    labels={"total_size_mb": "크기 (MB)", "category_label": "카테고리"},
    hover_data=["file_count", "category"],
    color="total_size_mb",
    color_continuous_scale="Blues",
)
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# ── 카테고리별 레코드 수 바 차트 (로그 스케일) ────────────
df_cat_records = (
    df_inv.groupby(["category", "category_label"])
    .agg(total_records=("record_count", "sum"), file_count=("filename", "count"))
    .reset_index()
    .sort_values("total_records", ascending=True)
)

fig = px.bar(
    df_cat_records,
    x="total_records",
    y="category_label",
    orientation="h",
    title="카테고리별 총 레코드 수 (로그 스케일)",
    labels={"total_records": "레코드 수", "category_label": "카테고리"},
    hover_data=["file_count", "category"],
    color="total_records",
    color_continuous_scale="Greens",
    log_x=True,
)
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# ── 파일 크기 vs 레코드 수 산점도 (버블 차트) ────────────
fig = px.scatter(
    df_inv,
    x="size_mb",
    y="record_count",
    size="size_mb",
    color="category_label",
    hover_name="filename",
    hover_data=["agency", "category"],
    title="파일 크기 vs 레코드 수 (버블: 파일 크기)",
    labels={"size_mb": "파일 크기 (MB)", "record_count": "레코드 수"},
    log_x=True,
    log_y=True,
    size_max=50,
)
fig.update_layout(height=600)
fig.show()

## 3. 스키마 발견

카테고리별 1,000건 샘플에서 필드명, 타입, 존재율을 분석합니다.

In [ ]:
# ── 카테고리별 스키마 분석 ─────────────────────────────────
schema_results = {}

for cat_key, cat_info in tqdm(CATEGORIES.items(), desc="스키마 분석"):
    # 카테고리의 첫 번째 파일에서 샘플 추출
    first_file = cat_info["files"][0]
    filepath = DATA_DIR / first_file
    if not filepath.exists():
        print(f"  [SKIP] {first_file} not found")
        continue

    sample = get_sample(filepath, n=1000)
    field_types = infer_field_types(sample)

    schema_results[cat_key] = {
        "label": cat_info["label"],
        "sample_file": first_file,
        "sample_count": len(sample),
        "field_count": len(field_types),
        "fields": {
            k: {
                "types": v["types"],
                "presence_rate": v["presence_rate"],
                "null_rate": v["null_rate"],
                "empty_rate": v["empty_rate"],
            }
            for k, v in field_types.items()
        },
    }
    print(f"  {cat_info['label']}: {len(field_types)}개 필드")

In [ ]:
# ── 스키마 패밀리 비교 테이블 (plotly table) ──────────────
table_rows = []
for cat_key, schema in schema_results.items():
    fields = schema["fields"]
    field_names = list(fields.keys())
    field_types_str = ", ".join(
        f"{k} ({'/'.join(v['types'])})" for k, v in list(fields.items())[:8]
    )
    if len(fields) > 8:
        field_types_str += f" ... +{len(fields) - 8}개"
    table_rows.append({
        "카테고리": schema["label"],
        "필드 수": schema["field_count"],
        "샘플 수": schema["sample_count"],
        "주요 필드": field_types_str,
    })

df_schema_table = pd.DataFrame(table_rows)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_schema_table.columns),
        fill_color="#4472C4",
        font=dict(color="white", size=12),
        align="left",
    ),
    cells=dict(
        values=[df_schema_table[col] for col in df_schema_table.columns],
        fill_color="#F2F2F2",
        align="left",
        font=dict(size=11),
        height=30,
    ),
)])
fig.update_layout(title="카테고리별 스키마 비교", height=max(400, len(table_rows) * 40 + 100))
fig.show()

In [ ]:
# ── 카테고리별 필드 수 비교 바 차트 ──────────────────────
df_field_counts = pd.DataFrame([
    {"category_label": s["label"], "field_count": s["field_count"]}
    for s in schema_results.values()
]).sort_values("field_count", ascending=True)

fig = px.bar(
    df_field_counts,
    x="field_count",
    y="category_label",
    orientation="h",
    title="카테고리별 필드 수 비교",
    labels={"field_count": "필드 수", "category_label": "카테고리"},
    color="field_count",
    color_continuous_scale="Purples",
)
fig.update_layout(height=500, showlegend=False)
fig.show()

In [ ]:
# ── 결과 저장 ─────────────────────────────────────────────
# Phase 1: 인벤토리
p1_path = save_result("phase1_inventory", inventory)
print(f"Phase 1 저장: {p1_path}")

# Phase 2: 스키마
p2_path = save_result("phase2_schema", schema_results)
print(f"Phase 2 저장: {p2_path}")

print(f"\n=== 요약 ===")
print(f"총 파일 수: {len(inventory)}")
print(f"총 크기: {sum(f['size_mb'] for f in inventory):.1f} MB")
print(f"총 레코드: {sum(f['record_count'] for f in inventory):,}건")
print(f"카테고리 수: {len(schema_results)}")